In [ ]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns


In [ ]:
# Load evaluation data
df = pd.read_csv("data.csv")
df = df.iloc[:99]  # Adjust as needed
# df = df.dropna()

# Rename columns for easier access
df = df.rename(columns={
    "context recall - measures how many of the relevant documents (or pieces of information) were successfully retrieved. [0, 5]": "context_recall",
    "faithfulness- נאמנות למידע שהוזן\nהאם התשובה נאמנה למה שנמצא במסמכים שנשלפו (בלי להמציא)?\n[0, 1]": "faithfulness",
    "answer relevance - עד כמה התשובה שהמודל נתן רלוונטית? [0, 1]": "relevance",
    "Rank - 1 best, 5 worst": "rank", 
    "Completeness - 0 - 5 (5-all data exist)": "completeness"
})


In [ ]:
# Convert to numeric, set errors='coerce' so invalid values -> NaN
df["rank"] = pd.to_numeric(df["rank"], errors="coerce")
df["rank"] = df["rank"].replace(0, np.nan).fillna(8).astype(int)

df["faithfulness"] = pd.to_numeric(df["faithfulness"], errors="coerce")
df["faithfulness"] = df["faithfulness"].fillna(0).astype(int)


In [ ]:
df["context_recall_norm"] = df["context_recall"] / df["context_recall"].max()
df["completeness_norm"] = df["completeness"] / df["completeness"].max()


In [ ]:
df.describe()


In [ ]:
ax = sns.histplot(data=df, x="faithfulness", kde=False, legend=False)
ax.set_xlabel("Faithfulness")
plt.title("Faithfulness Histogram")
plt.show()


In [ ]:
plt.scatter(df["context_recall"], df["rank"])
plt.xlabel("Context Recall")
plt.ylabel("Rank")
plt.title("Context Recall vs Rank")
plt.show()


In [ ]:
# Filter out rows with URLs in Ground Truth (if needed)
df = df[~df["Ground Truth"].str.contains("https", na=False)].dropna()
df


In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# Create a function to compute cosine similarity between two strings
def compute_cosine_similarity(a, b):
    vect = TfidfVectorizer().fit([a, b])   # fit on both strings
    tfidf = vect.transform([a, b])         # transform both strings
    return cosine_similarity(tfidf[0], tfidf[1])[0,0]

# Apply row by row
df["cosine_similarity"] = df.apply(lambda row: compute_cosine_similarity(row["Ground Truth"], row["gemini 2.5 flash response"]), axis=1)
df[["Ground Truth", "gemini 2.5 flash response", "cosine_similarity"]]


In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

COL_GT   = "Ground Truth"
COL_RESP = "gemini 2.5 flash response"
N        = 100   # number of queries (rows) to visualize in the heatmap
MODEL_ID = "intfloat/multilingual-e5-large"  # matches your paper setup
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"

# Keep only the first N examples (and drop rows with missing text)
df_subset = (
    df[[COL_GT, COL_RESP]]
    .dropna(subset=[COL_GT, COL_RESP])
    .head(N)
    .reset_index(drop=True)
)

# Clean text function
def clean_txt(s: str) -> str:
    s = str(s).strip()
    return s.replace("\u200f", "").replace("\u200e", "").replace("\u202a", "").replace("\u202c", "")

gt_texts   = [clean_txt(t) for t in df_subset[COL_GT].tolist()]
resp_texts = [clean_txt(t) for t in df_subset[COL_RESP].tolist()]

# E5 expects a task prefix; for content passages, "passage: " is appropriate
def add_prefix(texts):
    return [f"passage: {t}" for t in texts]

model = SentenceTransformer(MODEL_ID, device=DEVICE)

with torch.inference_mode():
    gt_embs   = model.encode(add_prefix(gt_texts), batch_size=32, convert_to_numpy=True, show_progress_bar=True)
    resp_embs = model.encode(add_prefix(resp_texts), batch_size=32, convert_to_numpy=True, show_progress_bar=True)


In [ ]:
# Full GT x Response matrix (N x N)
sim_matrix = cosine_similarity(gt_embs, resp_embs)

# Diagonal = similarity between each query's GT and its own response
diag_sim = np.diag(sim_matrix)

# Heatmap of GT x Response similarities
plt.figure(figsize=(8, 6))
plt.imshow(sim_matrix, aspect="auto")
plt.colorbar(label="Cosine similarity")
plt.xticks(ticks=np.arange(sim_matrix.shape[1]), labels=[f"Resp {i}" for i in range(sim_matrix.shape[1])], rotation=90)
plt.yticks(ticks=np.arange(sim_matrix.shape[0]), labels=[f"GT {i}" for i in range(sim_matrix.shape[0])])
plt.title(f"GT × Response Similarity (first {sim_matrix.shape[0]} queries)")
plt.xlabel("Responses")
plt.ylabel("Ground Truths")
plt.tight_layout()
plt.show()

# Line chart of the diagonal (pairwise GT vs. its own response)
plt.figure(figsize=(8, 3.5))
plt.scatter(range(len(diag_sim)), diag_sim, marker="o")
plt.axhline(y=0.85, color="red", linestyle="--", linewidth=1)
plt.title("Pairwise GT–Response Similarity")
plt.xlabel("Query index")
plt.ylabel("Cosine similarity")
plt.grid(True, linewidth=0.3)
plt.tight_layout()
plt.show()

# Print a small table of the first few pairwise similarities
pairwise_df = pd.DataFrame({
    "idx": range(len(diag_sim)),
    "gt_resp_cosine": np.round(diag_sim, 3),
})
print(pairwise_df.head(10).to_string(index=False))
